In [1]:
# 1. Verify GPU
import torch, sys
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU: Runtime > Change runtime type > T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("Free GPU memory (GB):", round(torch.cuda.mem_get_info()[0] / 1024**3, 2))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
CUDA: 13.0
CUDA available: True
GPU: Tesla T4
Free GPU memory (GB): 14.46


In [3]:
# 2. Clone once, otherwise pull the latest committed source
from pathlib import Path
if not Path("/content/Legal-Clause-Extraction-CUAD").exists():
    !git clone https://github.com/ArwaSaluji01/Legal-Clause-Extraction-CUAD.git
%cd /content/Legal-Clause-Extraction-CUAD
!git pull origin main


/content/Legal-Clause-Extraction-CUAD
From https://github.com/ArwaSaluji01/Legal-Clause-Extraction-CUAD
 * branch            main       -> FETCH_HEAD
Already up to date.


In [ ]:
%pip install -q --upgrade \
    "numpy==2.2.6" \
    "scipy>=1.15.2,<2" \
    "scikit-learn==1.7.2" \
    "transformers==4.46.3" \
    "huggingface-hub>=0.23.2,<1.0" \
    "safetensors>=0.4.5" \
    "requests==2.32.4" \
    "tqdm==4.67.1"

In [2]:
# Verify the repaired environment

import sys
import torch
import numpy as np
import scipy
import sklearn
import transformers
import requests

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Transformers:", transformers.__version__)
print("Requests:", requests.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU is unavailable. Select T4 GPU in Runtime settings.")

print("GPU:", torch.cuda.get_device_name(0))

# Verify that the previously failing imports work.
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

print("\nPASS: All required imports work.")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
CUDA: 13.0
NumPy: 2.2.6
SciPy: 1.18.1
Scikit-learn: 1.7.2
Transformers: 4.46.3
Requests: 2.32.4

CUDA available: True
GPU: Tesla T4

PASS: All required imports work.


In [4]:
# 5. Download and prepare CUAD only if processed files are absent
from pathlib import Path
required = [
    Path("data/processed/train.jsonl"),
    Path("data/processed/validation.jsonl"),
    Path("data/processed/test.jsonl"),
]
if not all(p.exists() for p in required):
    !python src/dataset/download_and_inspect.py
    !python src/dataset/prepare_dataset.py
else:
    print("Processed CUAD files already exist; skipping download/preparation.")
!python src/dataset/validate_processed.py

Source: https://github.com/TheAtticusProject/cuad/raw/main/data.zip

SHA256 verification:
Expected: f8161d18bea4e9c05e78fa6dda61c19c846fb8087ea969c172753bc2f45b999a
Actual  : f8161d18bea4e9c05e78fa6dda61c19c846fb8087ea969c172753bc2f45b999a

Dataset downloaded and checksum verified.

Extracting dataset...
Extraction completed.

CUAD ARCHIVE FILE INSPECTION
CUADv1.json (39188.12 KB)
test.json (7205.30 KB)
train_separate_questions.json (34243.99 KB)

JSON FILE: CUADv1.json
Object with 2 keys
version:
  String | length=8 | preview='aok_v1.0'
data:
  Array with 510 elements
  First element:
    Object with 2 keys
    title:
      String | length=51 | preview='LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT'
    paragraphs:
      Array with 1 elements
      First element:
        Object with 2 keys
        qas:
          ... maximum inspection depth reached
        context:
          ... maximum inspection depth reached

JSON FILE: test.json
Object with 2 keys
version:
  String | length=

In [5]:
# 6. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
# 7. Store checkpoints, logs and reduced feature caches on Google Drive
from pathlib import Path
import shutil

drive_root = Path("/content/drive/MyDrive/CUAD_Project")
drive_checkpoints = drive_root / "checkpoints"
drive_logs = drive_root / "logs"
drive_checkpoints.mkdir(parents=True, exist_ok=True)
drive_logs.mkdir(parents=True, exist_ok=True)

local_checkpoints = Path("models/checkpoints")
local_logs = Path("logs")
local_checkpoints.parent.mkdir(parents=True, exist_ok=True)

for local_path, drive_path in [
    (local_checkpoints, drive_checkpoints),
    (local_logs, drive_logs),
]:
    if local_path.is_symlink():
        local_path.unlink()
    elif local_path.exists():
        # Preserve any existing local files before replacing the directory.
        if any(local_path.iterdir()):
            backup = local_path.with_name(local_path.name + "_local_backup")
            if backup.exists():
                shutil.rmtree(backup)
            local_path.rename(backup)
        else:
            local_path.rmdir()
    local_path.symlink_to(drive_path, target_is_directory=True)

print("Checkpoints/cache:", drive_checkpoints)
print("Logs:", drive_logs)

Checkpoints/cache: /content/drive/MyDrive/CUAD_Project/checkpoints
Logs: /content/drive/MyDrive/CUAD_Project/logs


In [7]:
# 8. GPU status and disk space
!nvidia-smi
!df -h /content

Sat Nov  7 07:57:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             31W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
%%writefile src/training/train.py
import argparse
import json
import random
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path(__file__).resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.config import (
    TRAIN_FILE,
    VALIDATION_FILE,
    MODEL_NAME,
    MODEL_MAX_LENGTH,
    DEVICE,
    LEARNING_RATE,
    WEIGHT_DECAY,
    MAX_GRAD_NORM,
    CHECKPOINT_DIR,
    LOGS_DIR,
    RANDOM_SEED,
)
from src.preprocessing.tokenizer import CUADTokenizer
from transformers import AutoModelForQuestionAnswering


# ---------------------------------------------------------
# REPRODUCIBILITY
# ---------------------------------------------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ---------------------------------------------------------
# DATA SAMPLING
# ---------------------------------------------------------

def shuffled_jsonl_records(file_path, limit, seed, buffer_size=512):
    """
    Bounded-buffer shuffle.
    Does not load the complete JSONL dataset into memory.
    """

    rng = random.Random(seed)
    buffer = []
    yielded = 0

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue

            buffer.append(json.loads(line))

            if len(buffer) >= buffer_size:
                index = rng.randrange(len(buffer))
                yield buffer.pop(index)
                yielded += 1

                if yielded >= limit:
                    return

    rng.shuffle(buffer)

    for record in buffer:
        if yielded >= limit:
            break

        yield record
        yielded += 1


# ---------------------------------------------------------
# FEATURE SELECTION
# ---------------------------------------------------------

def select_features_for_example(features, rng):
    """
    Answerable example:
      - one positive window containing the answer
      - at most one negative window

    Unanswerable example:
      - one randomly selected window

    An answerable example without a positive window is
    excluded and audited by the caller.
    """

    if not features:
        return [], "empty"

    positive = [
        feature
        for feature in features
        if (
            feature["start_positions"] != feature["cls_index"]
            or feature["end_positions"] != feature["cls_index"]
        )
    ]

    negative = [
        feature
        for feature in features
        if (
            feature["start_positions"] == feature["cls_index"]
            and feature["end_positions"] == feature["cls_index"]
        )
    ]

    is_answerable = (
        features[0].get("answer_text") is not None
        or any(
            feature["start_positions"] != feature["cls_index"]
            for feature in features
        )
    )

    if is_answerable:
        if not positive:
            return [], "unaligned_answer"

        selected = [rng.choice(positive)]

        if negative:
            selected.append(rng.choice(negative))

        return selected, "answerable"

    return [rng.choice(features)], "unanswerable"


# ---------------------------------------------------------
# CACHE GENERATION
# ---------------------------------------------------------

def build_feature_cache(
    file_path,
    cache_path,
    tokenizer,
    max_examples,
    seed,
    max_features,
):
    """
    Tokenizes a reproducible sample and saves only selected
    training windows. Full-document tokenization is retained
    so that answer locations are found correctly.
    """

    cache_path.parent.mkdir(parents=True, exist_ok=True)

    if cache_path.exists():
        print(f"Loading existing feature cache: {cache_path}")

        cached = np.load(cache_path)

        arrays = {
            key: cached[key]
            for key in (
                "input_ids",
                "attention_mask",
                "start_positions",
                "end_positions",
            )
        }

        print("Cached features:", len(arrays["input_ids"]))
        return arrays

    rng = random.Random(seed)

    input_ids = []
    attention_masks = []
    start_positions = []
    end_positions = []

    audit = {
        "examples_seen": 0,
        "answerable_examples": 0,
        "unanswerable_examples": 0,
        "unaligned_answer_examples": [],
        "empty_examples": 0,
        "features_kept": 0,
        "categories": {},
    }

    print(f"\nBuilding feature cache: {file_path}")
    print(f"Examples to sample: {max_examples}")

    for example in shuffled_jsonl_records(
        file_path,
        limit=max_examples,
        seed=seed,
    ):
        audit["examples_seen"] += 1

        features = tokenizer.tokenize_example(example)

        # Attach answer information for safe classification.
        answer_text, _, _ = tokenizer._get_answer(example)

        for feature in features:
            feature["answer_text"] = answer_text

        selected, status = select_features_for_example(
            features,
            rng,
        )

        category = example.get("category", "UNKNOWN")
        audit["categories"][category] = (
            audit["categories"].get(category, 0) + 1
        )

        if status == "unaligned_answer":
            audit["unaligned_answer_examples"].append(
                example.get("id", "UNKNOWN")
            )
            continue

        if status == "empty":
            audit["empty_examples"] += 1
            continue

        if status == "answerable":
            audit["answerable_examples"] += 1
        else:
            audit["unanswerable_examples"] += 1

        for feature in selected:
            if len(input_ids) >= max_features:
                break

            input_ids.append(feature["input_ids"])
            attention_masks.append(feature["attention_mask"])
            start_positions.append(feature["start_positions"])
            end_positions.append(feature["end_positions"])

        if audit["examples_seen"] % 100 == 0:
            print(
                f"Examples processed: {audit['examples_seen']} | "
                f"Selected features: {len(input_ids)} | "
                f"Unaligned answers: "
                f"{len(audit['unaligned_answer_examples'])}"
            )

        if len(input_ids) >= max_features:
            break

    if not input_ids:
        raise RuntimeError("No usable features were generated.")

    arrays = {
        "input_ids": np.asarray(input_ids, dtype=np.uint16),
        "attention_mask": np.asarray(attention_masks, dtype=np.uint8),
        "start_positions": np.asarray(start_positions, dtype=np.uint16),
        "end_positions": np.asarray(end_positions, dtype=np.uint16),
    }

    np.savez_compressed(cache_path, **arrays)

    audit_path = cache_path.with_name(
        cache_path.stem + "_audit.json"
    )

    with open(audit_path, "w", encoding="utf-8") as file:
        json.dump(audit, file, indent=2)

    print("\nFeature cache completed.")
    print("Examples processed:", audit["examples_seen"])
    print("Answerable examples:", audit["answerable_examples"])
    print("Unanswerable examples:", audit["unanswerable_examples"])
    print("Unaligned answer examples:", len(audit["unaligned_answer_examples"]))
    print("Features retained:", len(input_ids))
    print("Categories represented:", len(audit["categories"]))
    print("Cache:", cache_path)
    print("Audit:", audit_path)

    return arrays


# ---------------------------------------------------------
# DATASET
# ---------------------------------------------------------

def make_dataloader(arrays, batch_size, shuffle, seed):
    dataset = TensorDataset(
        torch.from_numpy(arrays["input_ids"].astype(np.int64)),
        torch.from_numpy(arrays["attention_mask"].astype(np.int64)),
        torch.from_numpy(arrays["start_positions"].astype(np.int64)),
        torch.from_numpy(arrays["end_positions"].astype(np.int64)),
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )


# ---------------------------------------------------------
# CHECKPOINTS
# ---------------------------------------------------------

def save_checkpoint(
    model,
    tokenizer,
    output_dir,
    epoch,
    next_batch,
    global_step,
    validation_loss=None,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        output_dir,
        safe_serialization=True,
    )

    tokenizer.tokenizer.save_pretrained(output_dir)

    metadata = {
        "epoch": epoch,
        "next_batch": next_batch,
        "global_step": global_step,
        "validation_loss": validation_loss,
        "model_name": MODEL_NAME,
        "max_length": MODEL_MAX_LENGTH,
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    with open(
        output_dir / "training_metadata.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(metadata, file, indent=2)

    print(f"\nCheckpoint saved: {output_dir}")


def load_resume_checkpoint(model, resume_dir):
    resume_dir = Path(resume_dir)

    metadata_path = resume_dir / "training_metadata.json"

    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Resume metadata not found: {metadata_path}"
        )

    with open(metadata_path, "r", encoding="utf-8") as file:
        metadata = json.load(file)

    resumed_model = AutoModelForQuestionAnswering.from_pretrained(
        resume_dir
    ).to(DEVICE)

    print("Resuming from:", resume_dir)
    print("Resume metadata:", metadata)

    return (
        resumed_model,
        int(metadata["epoch"]),
        int(metadata["next_batch"]),
        int(metadata["global_step"]),
    )


# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------

def evaluate_loss(model, dataloader, use_amp):
    model.eval()

    total_loss = 0.0
    total_examples = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids, attention_mask, starts, ends = [
                tensor.to(DEVICE, non_blocking=True)
                for tensor in batch
            ]

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp,
            ):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    start_positions=starts,
                    end_positions=ends,
                )

            batch_count = input_ids.size(0)
            total_loss += outputs.loss.item() * batch_count
            total_examples += batch_count

    if total_examples == 0:
        raise RuntimeError("Validation dataset is empty.")

    return total_loss / total_examples


# ---------------------------------------------------------
# TRAINING
# ---------------------------------------------------------

def train(args):
    set_seed(RANDOM_SEED)

    use_amp = torch.cuda.is_available()

    print("=" * 65)
    print("CUAD TRAINING")
    print("=" * 65)
    print("Device:", DEVICE)
    print("GPU:", torch.cuda.get_device_name(0) if use_amp else "None")
    print("Model:", MODEL_NAME)
    print("Epochs:", args.epochs)
    print("Train examples:", args.train_examples)
    print("Validation examples:", args.validation_examples)
    print("Batch size:", args.batch_size)
    print("Gradient accumulation:", args.gradient_accumulation)
    print("Mixed precision:", use_amp)

    drive_root = Path("/content/drive/MyDrive/CUAD_Project")

    if drive_root.exists():
        cache_dir = drive_root / "cache"
    else:
        cache_dir = PROJECT_ROOT / "cache"

    cache_dir.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    LOGS_DIR.mkdir(parents=True, exist_ok=True)

    train_cache = cache_dir / f"train_{args.train_examples}_v1.npz"
    val_cache = cache_dir / f"validation_{args.validation_examples}_v1.npz"

    print("\nLoading tokenizer...")
    tokenizer = CUADTokenizer()

    train_arrays = build_feature_cache(
        TRAIN_FILE,
        train_cache,
        tokenizer,
        args.train_examples,
        RANDOM_SEED,
        args.max_train_features,
    )

    val_arrays = build_feature_cache(
        VALIDATION_FILE,
        val_cache,
        tokenizer,
        args.validation_examples,
        RANDOM_SEED + 1,
        args.max_validation_features,
    )

    train_loader = make_dataloader(
        train_arrays,
        args.batch_size,
        shuffle=True,
        seed=RANDOM_SEED,
    )

    val_loader = make_dataloader(
        val_arrays,
        args.batch_size,
        shuffle=False,
        seed=RANDOM_SEED,
    )

    print("\nTraining features:", len(train_arrays["input_ids"]))
    print("Validation features:", len(val_arrays["input_ids"]))
    print("Training batches per epoch:", len(train_loader))

    if args.resume:
        model, start_epoch, resume_batch, global_step = (
            load_resume_checkpoint(
                None,
                args.resume,
            )
        )
    else:
        print("\nLoading pretrained QA model...")
        model = AutoModelForQuestionAnswering.from_pretrained(
            MODEL_NAME
        ).to(DEVICE)

        start_epoch = 1
        resume_batch = 0
        global_step = 0

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp,
    )

    latest_dir = CHECKPOINT_DIR / "latest_step"
    best_dir = CHECKPOINT_DIR / "best_model"

    best_validation_loss = float("inf")

    if args.resume:
        metadata_path = Path(args.resume) / "training_metadata.json"
        with open(metadata_path, "r", encoding="utf-8") as file:
            resume_metadata = json.load(file)

        previous_loss = resume_metadata.get("best_validation_loss")
        if previous_loss is not None:
            best_validation_loss = float(previous_loss)

    history_file = LOGS_DIR / "training_history.jsonl"

    for epoch in range(start_epoch, args.epochs + 1):
        print("\n" + "=" * 65)
        print(f"EPOCH {epoch}/{args.epochs}")
        print("=" * 65)

        model.train()
        optimizer.zero_grad(set_to_none=True)

        total_loss = 0.0
        seen_features = 0
        pending_gradients = 0

        epoch_loader = make_dataloader(
            train_arrays,
            args.batch_size,
            shuffle=True,
            seed=RANDOM_SEED + epoch,
        )

        skip_batches = resume_batch if epoch == start_epoch else 0

        if skip_batches:
            print("Skipping already completed batches:", skip_batches)

        for batch_index, batch in enumerate(epoch_loader):
            if batch_index < skip_batches:
                continue

            input_ids, attention_mask, starts, ends = [
                tensor.to(DEVICE, non_blocking=True)
                for tensor in batch
            ]

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp,
            ):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    start_positions=starts,
                    end_positions=ends,
                )
                loss = outputs.loss

            scaler.scale(
                loss / args.gradient_accumulation
            ).backward()

            batch_count = input_ids.size(0)
            total_loss += loss.item() * batch_count
            seen_features += batch_count
            pending_gradients += 1

            is_last_batch = (
                batch_index + 1 == len(epoch_loader)
            )

            if (
                pending_gradients == args.gradient_accumulation
                or is_last_batch
            ):
                # Correct the final partial accumulation group.
                if pending_gradients < args.gradient_accumulation:
                    correction = (
                        args.gradient_accumulation / pending_gradients
                    )

                    for parameter in model.parameters():
                        if parameter.grad is not None:
                            parameter.grad.mul_(correction)

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM,
                )

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                pending_gradients = 0
                global_step += 1

                if global_step % 25 == 0:
                    average_loss = total_loss / max(seen_features, 1)

                    print(
                        f"Epoch {epoch} | "
                        f"Step {global_step} | "
                        f"Features {seen_features}/{len(train_arrays['input_ids'])} | "
                        f"Loss {average_loss:.4f}"
                    )

                if (
                    global_step % args.checkpoint_every == 0
                    and not is_last_batch
                ):
                    save_checkpoint(
                        model,
                        tokenizer,
                        latest_dir,
                        epoch,
                        batch_index + 1,
                        global_step,
                    )

                    # Store best loss in the latest checkpoint metadata.
                    metadata_path = latest_dir / "training_metadata.json"
                    with open(metadata_path, "r", encoding="utf-8") as file:
                        metadata = json.load(file)

                    metadata["best_validation_loss"] = best_validation_loss

                    with open(metadata_path, "w", encoding="utf-8") as file:
                        json.dump(metadata, file, indent=2)

        if seen_features == 0:
            raise RuntimeError(
                "No training features were processed in this epoch."
            )

        train_loss = total_loss / seen_features

        validation_loss = evaluate_loss(
            model,
            val_loader,
            use_amp,
        )

        print("\nEpoch summary")
        print("Training features:", seen_features)
        print(f"Training loss: {train_loss:.4f}")
        print(f"Validation loss: {validation_loss:.4f}")

        record = {
            "epoch": epoch,
            "training_features": seen_features,
            "training_loss": train_loss,
            "validation_features": len(val_arrays["input_ids"]),
            "validation_loss": validation_loss,
            "global_step": global_step,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        with open(history_file, "a", encoding="utf-8") as file:
            file.write(json.dumps(record) + "\n")

        epoch_dir = CHECKPOINT_DIR / f"epoch_{epoch}"

        save_checkpoint(
            model,
            tokenizer,
            epoch_dir,
            epoch,
            0,
            global_step,
            validation_loss,
        )

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss

            save_checkpoint(
                model,
                tokenizer,
                best_dir,
                epoch,
                0,
                global_step,
                validation_loss,
            )

        # Update the emergency checkpoint to mark this epoch complete.
        save_checkpoint(
            model,
            tokenizer,
            latest_dir,
            epoch + 1,
            0,
            global_step,
            validation_loss,
        )

        metadata_path = latest_dir / "training_metadata.json"
        with open(metadata_path, "r", encoding="utf-8") as file:
            metadata = json.load(file)

        metadata["best_validation_loss"] = best_validation_loss

        with open(metadata_path, "w", encoding="utf-8") as file:
            json.dump(metadata, file, indent=2)

        # The next epoch starts from its first batch.
        resume_batch = 0

    print("\n" + "=" * 65)
    print("TRAINING FINISHED")
    print("=" * 65)
    print("Best validation loss:", best_validation_loss)
    print("Best model:", best_dir)
    print("Training history:", history_file)
    print("Feature cache:", cache_dir)


# ---------------------------------------------------------
# COMMAND LINE
# ---------------------------------------------------------

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--train-examples", type=int, default=4000)
    parser.add_argument("--validation-examples", type=int, default=400)
    parser.add_argument("--max-train-features", type=int, default=8000)
    parser.add_argument("--max-validation-features", type=int, default=800)
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--gradient-accumulation", type=int, default=2)
    parser.add_argument("--checkpoint-every", type=int, default=250)
    parser.add_argument("--resume", type=str, default=None)
    parser.add_argument("--rebuild-cache", action="store_true")

    args = parser.parse_args()

    if args.epochs < 1:
        raise ValueError("Epochs must be at least 1.")

    if args.batch_size < 1 or args.gradient_accumulation < 1:
        raise ValueError("Batch size and accumulation must be positive.")

    if args.checkpoint_every < 1:
        raise ValueError("Checkpoint interval must be positive.")

    drive_root = Path("/content/drive/MyDrive/CUAD_Project")
    cache_dir = (
        drive_root / "cache"
        if drive_root.exists()
        else PROJECT_ROOT / "cache"
    )

    if args.rebuild_cache:
        for cache_file in (
            cache_dir / f"train_{args.train_examples}_v1.npz",
            cache_dir / f"validation_{args.validation_examples}_v1.npz",
        ):
            if cache_file.exists():
                cache_file.unlink()
                print("Removed cache:", cache_file)

    train(args)


if __name__ == "__main__":
    main()

Overwriting src/training/train.py


In [10]:
%cd /content/Legal-Clause-Extraction-CUAD

/content/Legal-Clause-Extraction-CUAD


In [11]:
!python -m py_compile src/training/train.py

In [13]:
!python -m src.training.train --help

usage: train.py [-h] [--epochs EPOCHS] [--train-examples TRAIN_EXAMPLES]
                [--validation-examples VALIDATION_EXAMPLES]
                [--max-train-features MAX_TRAIN_FEATURES]
                [--max-validation-features MAX_VALIDATION_FEATURES]
                [--batch-size BATCH_SIZE]
                [--gradient-accumulation GRADIENT_ACCUMULATION]
                [--checkpoint-every CHECKPOINT_EVERY] [--resume RESUME]
                [--rebuild-cache]

options:
  -h, --help            show this help message and exit
  --epochs EPOCHS
  --train-examples TRAIN_EXAMPLES
  --validation-examples VALIDATION_EXAMPLES
  --max-train-features MAX_TRAIN_FEATURES
  --max-validation-features MAX_VALIDATION_FEATURES
  --batch-size BATCH_SIZE
  --gradient-accumulation GRADIENT_ACCUMULATION
  --checkpoint-every CHECKPOINT_EVERY
  --resume RESUME
  --rebuild-cache


In [14]:
from pathlib import Path

assert Path("data/processed/train.jsonl").exists()
assert Path("data/processed/validation.jsonl").exists()
assert Path("models/checkpoints").exists()

print("PASS: Training script and dataset paths are ready.")

PASS: Training script and dataset paths are ready.


In [ ]:
# 9. Train or resume
!python -u -m src.training.train \
    --epochs 2 \
    --train-examples 4000 \
    --validation-examples 400 \
    --max-train-features 8000 \
    --max-validation-features 800 \
    --batch-size 4 \
    --gradient-accumulation 2 \
    --checkpoint-every 250

In [16]:
# 10. Inspect training history
from pathlib import Path
history = Path("logs/training_history.jsonl")
if history.exists():
    for line in history.read_text(encoding="utf-8").splitlines():
        print(line)
else:
    print("Training history not found.")

{"epoch": 1, "training_features": 5749, "training_loss": 0.9867210927725377, "validation_features": 564, "validation_loss": 0.9855572703882312, "global_step": 719, "timestamp_utc": "2026-11-07T08:19:51.375062+00:00"}
{"epoch": 2, "training_features": 5749, "training_loss": 0.6969978763323532, "validation_features": 564, "validation_loss": 1.0385626417102543, "global_step": 1438, "timestamp_utc": "2026-11-07T08:22:51.058232+00:00"}


In [17]:
# 11. Validate the saved best checkpoint
!python -m src.training.validate_checkpoint

STAGE 4 CHECKPOINT VALIDATION
Training metadata:
{
    "epoch": 1,
    "next_batch": 0,
    "global_step": 719,
    "validation_loss": 0.9855572703882312,
    "model_name": "deepset/roberta-base-squad2",
    "max_length": 384,
    "saved_at_utc": "2026-11-07T08:19:59.249773+00:00"
}
I0000 00:00:1794039834.864414   15045 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794039838.120243   15045 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

PASS: Checkpoint loaded successfully.
PASS: Fast tokenizer loaded successfully.
PASS: Start and end logits have expected shapes.
PASS: Model runs on cuda.

ALL STAGE 4 VALIDATION CHECKS PASSED


In [18]:
# 12. Run a 100-example official-test diagnostic
!python -m src.evaluation.evaluate --limit 100

CUAD TEST SET EVALUATION
Device: cuda
Checkpoint: /content/Legal-Clause-Extraction-CUAD/models/checkpoints/best_model
Test example limit: 100
No-answer threshold: 0.0

Loading tokenizer...
Loading model...
I0000 00:00:1794039852.198550   15142 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794039855.289101   15142 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

Loading test examples...
Examples loaded: 100
[1/100] Category: Document Name | Chunks: 11 | EM: 0 | F1: 0.5000
[2/100] Category: Parties | Chunks: 12 | EM: 0 | F1: 0.0000
[3/100] Category: Agreement Date | Chunks: 12 | EM: 0.0 | F1: 0.0000
[4/100] Category: Effective Date | Chunks: 12 | EM: 0.0 | F1: 0.0000
[5/100] Category: Expiration Date | Chunks: 12 | EM: 1 |

In [ ]:
# 13. Confirm artifacts are on Google Drive
from pathlib import Path
root = Path("/content/drive/MyDrive/CUAD_Project")
for item in sorted(root.rglob("*")):
    if item.is_file():
        print(item, f"({item.stat().st_size / 1024**2:.1f} MB)")

In [24]:
from pathlib import Path
import json

drive = Path("/content/drive/MyDrive/CUAD_Project")

required = {
    "Best model": drive / "checkpoints/best_model",
    "Model weights": drive / "checkpoints/best_model/model.safetensors",
    "Model config": drive / "checkpoints/best_model/config.json",
    "Tokenizer": drive / "checkpoints/best_model/tokenizer.json",
    "Training history": drive / "logs/training_history.jsonl",
    "Evaluation metrics": drive / "results/test_metrics.json",
    "Test predictions": drive / "results/test_predictions.jsonl",
}

for name, path in required.items():
    status = "PASS" if path.exists() else "MISSING"
    print(f"{status}: {name} — {path}")

if (drive / "checkpoints/best_model/training_metadata.json").exists():
    metadata = json.loads(
        (drive / "checkpoints/best_model/training_metadata.json").read_text()
    )
    print("\nBest checkpoint metadata:")
    print(json.dumps(metadata, indent=2))

PASS: Best model — /content/drive/MyDrive/CUAD_Project/checkpoints/best_model
PASS: Model weights — /content/drive/MyDrive/CUAD_Project/checkpoints/best_model/model.safetensors
PASS: Model config — /content/drive/MyDrive/CUAD_Project/checkpoints/best_model/config.json
PASS: Tokenizer — /content/drive/MyDrive/CUAD_Project/checkpoints/best_model/tokenizer.json
PASS: Training history — /content/drive/MyDrive/CUAD_Project/logs/training_history.jsonl
MISSING: Evaluation metrics — /content/drive/MyDrive/CUAD_Project/results/test_metrics.json
MISSING: Test predictions — /content/drive/MyDrive/CUAD_Project/results/test_predictions.jsonl

Best checkpoint metadata:
{
  "epoch": 1,
  "next_batch": 0,
  "global_step": 719,
  "validation_loss": 0.9855572703882312,
  "model_name": "deepset/roberta-base-squad2",
  "max_length": 384,
  "saved_at_utc": "2026-11-07T08:19:59.249773+00:00"
}


In [25]:
from google.colab import files
files.download("src/training/train.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
# 14. Run the complete official test evaluation
from pathlib import Path
import shutil

project = Path("/content/Legal-Clause-Extraction-CUAD")
drive = Path("/content/drive/MyDrive/CUAD_Project")

# Preserve the earlier 100-example diagnostic
preliminary = drive / "results/preliminary_100_examples"
preliminary.mkdir(parents=True, exist_ok=True)

for filename in ["test_metrics.json", "test_predictions.jsonl"]:
    source = project / "results" / filename
    if source.exists():
        shutil.copy2(source, preliminary / filename)

# Verify test set size
test_file = project / "data/processed/test.jsonl"
with test_file.open("r", encoding="utf-8") as f:
    test_count = sum(1 for line in f if line.strip())

checkpoint = drive / "checkpoints/best_model"

print("Official test examples:", test_count)
print("Best checkpoint exists:", checkpoint.exists())
print("Model weights exist:", (checkpoint / "model.safetensors").exists())

assert test_count == 4182, f"Unexpected test count: {test_count}"
assert (checkpoint / "model.safetensors").exists()

Official test examples: 4182
Best checkpoint exists: True
Model weights exist: True


In [27]:
%cd /content/Legal-Clause-Extraction-CUAD

!python -u -m src.evaluation.evaluate \
    --limit 0 \
    --threshold 0.0 \
    --checkpoint /content/drive/MyDrive/CUAD_Project/checkpoints/best_model

/content/Legal-Clause-Extraction-CUAD
CUAD TEST SET EVALUATION
Device: cuda
Checkpoint: /content/drive/MyDrive/CUAD_Project/checkpoints/best_model
Test example limit: 0
No-answer threshold: 0.0

Loading tokenizer...
Loading model...
I0000 00:00:1794040821.132906   19418 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794040823.688578   19418 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

Loading test examples...
Examples loaded: 4182
[1/4182] Category: Document Name | Chunks: 11 | EM: 0 | F1: 0.5000
[2/4182] Category: Parties | Chunks: 12 | EM: 0 | F1: 0.0000
[3/4182] Category: Agreement Date | Chunks: 12 | EM: 0.0 | F1: 0.0000
[4/4182] Category: Effective Date | Chunks: 12 | EM: 0.0 | F1: 0.0000
[5/4182] Category: Expir

In [28]:
from pathlib import Path
import shutil

project = Path("/content/Legal-Clause-Extraction-CUAD")
drive = Path("/content/drive/MyDrive/CUAD_Project")

for filename in ["test_metrics.json", "test_predictions.jsonl"]:
    source = project / "results" / filename
    assert source.exists(), f"Missing output: {source}"
    shutil.copy2(source, drive / "results" / filename)
    print("Backed up:", filename)

print("Full evaluation artifacts saved to Google Drive.")

Backed up: test_metrics.json
Backed up: test_predictions.jsonl
Full evaluation artifacts saved to Google Drive.
